# 03 · Entrenar una red SIN backpropagation
Cada partícula = todos los pesos de la red. Solo forward. Cero derivadas.


In [ ]:
# Motor ABC + PSO (incluido para que Colab no dependa de rutas)
#!/usr/bin/env python3
"""Primitivas ABC y PSO usadas por los cuatro ejercicios."""

import numpy as np


def pso_minimize(objective, bounds, n_particles=20, iters=25, w=0.7, c1=1.5, c2=1.5, seed=42):
    """PSO global-best. Minimiza objective(x).

    Ciclo: representacion = posicion continua,
    inicializacion uniforme, aptitud = objective,
    comportamiento = inercia + pbest + gbest,
    evolucion por iteraciones, parada = iters.
    """
    rng = np.random.default_rng(seed)
    lo, hi = np.asarray(bounds[0], float), np.asarray(bounds[1], float)
    dim = lo.size
    pos = rng.uniform(lo, hi, size=(n_particles, dim))
    vel = np.zeros_like(pos)
    costs = np.array([objective(p) for p in pos])
    pbest, pbest_c = pos.copy(), costs.copy()
    g = int(np.argmin(pbest_c))
    gbest, gbest_c = pbest[g].copy(), float(pbest_c[g])
    hist = [gbest_c]
    swarm_hist = [pos.copy()]
    for _ in range(iters):
        r1, r2 = rng.random(pos.shape), rng.random(pos.shape)
        vel = w * vel + c1 * r1 * (pbest - pos) + c2 * r2 * (gbest - pos)
        pos = np.clip(pos + vel, lo, hi)
        costs = np.array([objective(p) for p in pos])
        improved = costs < pbest_c
        pbest[improved] = pos[improved]
        pbest_c[improved] = costs[improved]
        g = int(np.argmin(pbest_c))
        if pbest_c[g] < gbest_c:
            gbest, gbest_c = pbest[g].copy(), float(pbest_c[g])
        hist.append(gbest_c)
        swarm_hist.append(pos.copy())
    return gbest, gbest_c, np.array(hist), swarm_hist


def abc_binary_maximize(objective, n_bits, n_bees=10, cycles=12, limit=4, seed=42):
    """ABC binario. Maximiza objective(bitstring).

    Ciclo: representacion = fuente de alimento binaria,
    inicializacion aleatoria, aptitud = objective,
    comportamiento = empleada / observadora / exploradora,
    evolucion por ciclos, parada = cycles.
    """
    rng = np.random.default_rng(seed)
    foods = rng.integers(0, 2, size=(n_bees, n_bits))
    empty = foods.sum(axis=1) == 0
    if empty.any():
        foods[empty, rng.integers(0, n_bits, size=int(empty.sum()))] = 1
    fit = np.array([objective(f) for f in foods])
    trials = np.zeros(n_bees, dtype=int)
    best_i = int(np.argmax(fit))
    best, best_f = foods[best_i].copy(), float(fit[best_i])
    hist = [best_f]

    def neighbor(src):
        k = int(rng.integers(0, n_bits))
        nxt = src.copy()
        nxt[k] ^= 1
        if nxt.sum() == 0:
            nxt[k] = 1
        return nxt

    for _ in range(cycles):
        for i in range(n_bees):
            cand = neighbor(foods[i])
            fc = objective(cand)
            if fc >= fit[i]:
                foods[i], fit[i], trials[i] = cand, fc, 0
            else:
                trials[i] += 1
        probs = fit - fit.min() + 1e-9
        probs = probs / probs.sum()
        for _o in range(n_bees):
            i = int(rng.choice(n_bees, p=probs))
            cand = neighbor(foods[i])
            fc = objective(cand)
            if fc >= fit[i]:
                foods[i], fit[i], trials[i] = cand, fc, 0
            else:
                trials[i] += 1
        for i in range(n_bees):
            if trials[i] >= limit:
                foods[i] = rng.integers(0, 2, size=n_bits)
                if foods[i].sum() == 0:
                    foods[i][int(rng.integers(0, n_bits))] = 1
                fit[i] = objective(foods[i])
                trials[i] = 0
        bi = int(np.argmax(fit))
        if fit[bi] > best_f:
            best, best_f = foods[bi].copy(), float(fit[bi])
        hist.append(best_f)
    return best, best_f, np.array(hist)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)

def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def relu(z):
    return np.maximum(0.0, z)

def pack_shapes(dims):
    specs, n = [], 0
    for a, b in zip(dims[:-1], dims[1:]):
        specs.append(((a, b), (b,)))
        n += a * b + b
    return specs, n

def unpack(vec, specs):
    params, i = [], 0
    for ws, bs in specs:
        nw = int(np.prod(ws))
        W = vec[i:i+nw].reshape(ws); i += nw
        B = vec[i:i+bs[0]]; i += bs[0]
        params.append((W, B))
    return params

def forward_mlp(X, params, last="softmax"):
    h = X
    for i, (W, B) in enumerate(params):
        h = h @ W + B
        if i < len(params) - 1:
            h = relu(h)
        else:
            h = softmax(h) if last == "softmax" else 1/(1+np.exp(-h))
    return h


In [ ]:
iris = load_iris()
X = StandardScaler().fit_transform(iris.data)
y = iris.target
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=SEED)

dims = [4, 8, 4, 3]
specs, n_dim = pack_shapes(dims)
print("arquitectura", dims, "parametros", n_dim)

def objective(w):
    pred = forward_mlp(Xtr, unpack(w, specs))
    return float(-np.mean(np.log(pred[np.arange(len(ytr)), ytr] + 1e-8)))

best, cost, hist, _ = pso_minimize(objective, (np.full(n_dim,-2.5), np.full(n_dim,2.5)),
                                   n_particles=18, iters=25, w=0.65, c1=1.6, c2=1.6, seed=SEED)
params = unpack(best, specs)
acc_tr = float((forward_mlp(Xtr, params).argmax(1) == ytr).mean())
acc_te = float((forward_mlp(Xte, params).argmax(1) == yte).mean())

mlp = MLPClassifier(hidden_layer_sizes=(8,4), activation="relu", max_iter=400,
                    random_state=SEED, solver="adam").fit(Xtr, ytr)
print(f"PSO  train={acc_tr:.4f} test={acc_te:.4f}")
print(f"Adam test={mlp.score(Xte, yte):.4f}")


In [ ]:
# XOR visual
Xor = np.array([[0.,0.],[0.,1.],[1.,0.],[1.,1.]])
yor = np.array([0.,1.,1.,0.])
dims_x = [2,4,1]
specs_x, ndx = pack_shapes(dims_x)

def obj_xor(w):
    p = forward_mlp(Xor, unpack(w, specs_x), last="sigmoid").ravel()
    return float(np.mean((p-yor)**2))

bestx, costx, _, _ = pso_minimize(obj_xor, (np.full(ndx,-4.), np.full(ndx,4.)),
                                  n_particles=16, iters=30, seed=1)
prx = unpack(bestx, specs_x)
print("XOR MSE", costx, "preds", np.round(forward_mlp(Xor, prx, last="sigmoid").ravel(), 3))

xx, yy = np.meshgrid(np.linspace(-0.3,1.3,160), np.linspace(-0.3,1.3,160))
zz = forward_mlp(np.c_[xx.ravel(), yy.ravel()], prx, last="sigmoid").reshape(xx.shape)
fig, ax = plt.subplots(figsize=(5,4.4))
ax.contourf(xx, yy, zz, levels=20, cmap="RdBu", vmin=0, vmax=1)
ax.scatter(Xor[:,0], Xor[:,1], c=yor, cmap="RdBu", s=80, edgecolor="k")
ax.set_title("XOR — MLP 2-4-1 con PSO"); plt.tight_layout(); plt.show()
